Circular restricted three-body problem. The coordinate frame is Earth‑centered inertial. Earth and Moon are modeled as point masses. The mass of the spacecraft is insignificance with respect to the Earth and Moon. Moon is on a circular orbit around Earth (no Sun, no J2). Newtonian gravity. Units are km, km/s, seconds.  Required libraries are NumPy, SciPy, and Matplotlib. The integrator of the equation of motion is the scipy solver solve_ivp. I'm interested in free-return trajectories like in the Artemis 2 Mission, but this simulation does not match a particular mission.

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib import rcParams
from IPython.display import HTML

In [ ]:
# Simulation parameters
nDays = 8.4

# Moon Initial position (angle in radians)
moon_initial_angle = 5*np.pi/6

# TLI parameters
v_mag = 10.9351 # 10.8 km/s 
flight_path_angle_deg = 0.0 # 0.0 degrees is a purely tangential burn

In [ ]:
# Gravitational parameters [km^3/s^2]
MU_EARTH = 398600.4418   
MU_MOON  = 4902.800066 

R_EARTH_EQ = 6378 # km
R_EARTH_MN = 6371 # km

TLI_alt_km = 185.0 # km above Earth's surface

# Mean Earth-Moon distance [km]
R_EM = 384400.0  

# Moon orbital period [s]
# T_MOON = 27.3 * 24 * 3600 # s
T_MOON = 27.321661 * 24 * 3600
N_MOON = 2 * np.pi / T_MOON # rad/s Angular velocity of the Moon's orbit around Earth

In [ ]:
def moon_state(t):
    """Circular Moon orbit around Earth in ECI-like frame."""
    
    th = N_MOON * t - moon_initial_angle
    x = R_EM * np.cos(th)
    y = R_EM * np.sin(th)
    z = 0.0

    vx = -R_EM * N_MOON * np.sin(th)
    vy =  R_EM * N_MOON * np.cos(th)
    vz = 0.0

    return np.array([x, y, z, vx, vy, vz])

In [ ]:
def eom_earth_moon(t, y):
    """
    Equations of motion for a spacecraft under Earth + Moon gravity.

    Parameters
    ----------
    t : float
        Time in seconds.
    y : array_like, shape (6,)
        State vector [rx, ry, rz, vx, vy, vz] where position is in km and
        velocity is in km/s.

    Returns
    -------
    ndarray, shape (6,)
        Time derivative of the state:
        [vx, vy, vz, ax, ay, az].

    Notes
    -----
    The acceleration is the sum of
        - Earth gravity: -MU_EARTH * r / |r|^3
        - Moon gravity:  -MU_MOON  * (r - r_Moon) / |r - r_Moon|^3

    The Moon position r_Moon is obtained from moon_state(t), assuming a
    circular lunar orbit in the Earth-centered inertial frame.
        """
    rx, ry, rz, vx, vy, vz = y
    r_vec = np.array([rx, ry, rz])
    r = np.linalg.norm(r_vec)

    # Moon state
    moon = moon_state(t)
    rM = moon[:3]
    r_rel = r_vec - rM
    r_rel_norm = np.linalg.norm(r_rel)

    # Accelerations
    aE = -MU_EARTH * r_vec / r**3
    aM = -MU_MOON  * r_rel / r_rel_norm**3

    ax, ay, az = aE + aM

    return np.array([vx, vy, vz, ax, ay, az])


In [ ]:
def initial_state_free_return():
    # Start near Earth
    r0 = np.array([ 0.0, R_EARTH_EQ + TLI_alt_km, 0.0])

    # Rough TLI-like velocity
    flight_path_angle = np.deg2rad(flight_path_angle_deg)

    v_dir = np.array([-1.0, 0.0, 0.0])
    v_dir /= np.linalg.norm(v_dir)

    v_tan = v_mag * np.cos(flight_path_angle)
    v_rad = v_mag * np.sin(flight_path_angle)

    v0 = v_tan * v_dir + v_rad * (r0 / np.linalg.norm(r0))

    return np.hstack((r0, v0))


In [ ]:
def propagate(y0, t0, tf, npts=5000):
    t_eval = np.linspace(t0, tf, npts)
    sol = solve_ivp(
        eom_earth_moon,
        (t0, tf),
        y0,
        t_eval=t_eval,
        rtol=1e-9,
        atol=1e-12
    )
    return sol.t, sol.y


In [ ]:
def plot_3d(t, Y):
    x, y, z = Y[0], Y[1], Y[2]
    moon_states = np.array([moon_state(ti) for ti in t])

    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection='3d')

    ax.plot(x, y, z, label='Spacecraft')
    ax.scatter(0, 0, 0, color='b', s=80, label='Earth')

    ax.plot(moon_states[:,0], moon_states[:,1], moon_states[:,2],
            '--', color='gray', label='Moon orbit')
    plt.axis('equal')
    ax.set_xlabel('x [km]')
    ax.set_ylabel('y [km]')
    ax.set_zlabel('z [km]')
    ax.set_title('3-Body Problem 3D View')
    ax.legend()
    ax.set_box_aspect([1,1,1])
    plt.show()


In [ ]:
def plot_xy(t, Y):
    x, y = Y[0], Y[1]
    moon_states = np.array([moon_state(ti) for ti in t])

    plt.figure(figsize=(7,7))
    plt.plot(x, y, label='Spacecraft')
    plt.plot(moon_states[:,0], moon_states[:,1], '--', color='gray', label='Moon orbit')
    plt.scatter(0, 0, color='b', s=80, label='Earth')

    plt.axis('equal')
    plt.xlabel('x [km]')
    plt.ylabel('y [km]')
    plt.title('3-Body Problem: Earth-Moon Plane View')
    plt.grid(True)
    plt.legend()
    plt.show()


In [ ]:
def analyze_flyby(t, Y):
    r_sc = Y[:3].T
    moon_states = np.array([moon_state(ti) for ti in t])
    r_moon = moon_states[:, :3]

    d_earth = np.linalg.norm(r_sc, axis=1)
    d_moon  = np.linalg.norm(r_sc - r_moon, axis=1)

    idx_moon = np.argmin(d_moon)
    idx_return = np.argmin(d_earth[len(t)//2:]) + len(t)//2

    idx_TLI = 0
    vel_TLI = np.linalg.norm(Y[3:6, idx_TLI])
    vel_moon = np.linalg.norm(Y[3:6, idx_moon])
    vel_return = np.linalg.norm(Y[3:6, idx_return])
    print(f"TLI perigee altitude: {d_earth[idx_TLI] - R_EARTH_EQ:.1f} km at t = {t[idx_TLI]/3600:.2f} h")
    print(f"TLI perigee velocity: {vel_TLI:.1f} km/s")
    print(f"Closest approach to Moon: {d_moon[idx_moon]:.1f} km at t = {t[idx_moon]/3600:.2f} h")
    print(f"Velocity at closest approach to Moon: {vel_moon:.1f} km/s")
    print(f"Return perigee altitude: {d_earth[idx_return] - R_EARTH_EQ:.1f} km at t = {t[idx_return]/3600:.2f} h")
    print(f"Return perigee velocity: {vel_return:.1f} km/s")



In [ ]:
y0 = initial_state_free_return()
t, Y = propagate(y0, 0, nDays*24*3600)

In [ ]:
analyze_flyby(t, Y)
plot_3d(t, Y)
plot_xy(t, Y)

In [ ]:
# --- Trajectory Animation ---
%matplotlib widget
rcParams['animation.embed_limit'] = 200000
# ------------------------------------------------------------
# 3D TIME-LAPSE ANIMATION FUNCTION
# ------------------------------------------------------------
def animate_3body_trajectory(
    t,
    X_Iner, Y_Iner, Z_Iner,
    X_m1, Y_m1, Z_m1,
    X_m2, Y_m2, Z_m2,
    trail_length=900,
    interval=0.1
):
    fig = plt.figure(figsize=(8, 8))
    ax = fig.add_subplot(111, projection='3d')

    # min and max for axes limits
    X_min = np.array([X_Iner.min(), X_m1.min(), X_m2.min()]).min()
    Y_min = np.array([Y_Iner.min(), Y_m1.min(), Y_m2.min()]).min()
    #Z_min = np.array([Z_Iner.min(), Z_m1.min(), Z_m2.min()]).min()

    X_max = np.array([X_Iner.max(), X_m1.max(), X_m2.max()]).max()
    Y_max = np.array([Y_Iner.max(), Y_m1.max(), Y_m2.max()]).max()
    #Z_max = np.array([Z_Iner.max(), Z_m1.max(), Z_m2.max()]).max()
    
    ax.set_xlim(X_min/1.1, X_max*1.1)
    ax.set_ylim(Y_min/1.1, Y_max*1.1)
    #ax.set_zlim(Z_min/1.1, Z_max*1.1)

    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")
    ax.set_title("3-Body Problem: Spacecraft Trajectory (3D Time-Lapse)")

    # Primary bodies
    body1, = ax.plot([X_m1[0]], [Y_m1[0]], [Z_m1[0]], 'bo', markersize=8, label="Earth")
    body2, = ax.plot([X_m2[0]], [Y_m2[0]], [Z_m2[0]], 'ko', markersize=6, label="Moon")

    # Spacecraft trajectory + point
    traj, = ax.plot([], [], [], 'r-', lw=1.5)
    point, = ax.plot([], [], [], 'ro', markersize=4)

    # Update function
    def update(frame):
        i0 = max(0, frame - trail_length)

        traj.set_data(X_Iner[i0:frame], Y_Iner[i0:frame])
        traj.set_3d_properties(Z_Iner[i0:frame])

        point.set_data([X_Iner[frame]], [Y_Iner[frame]])
        point.set_3d_properties([Z_Iner[frame]])

        # Update moving primaries
        body1.set_data([X_m1[frame]], [Y_m1[frame]])
        body1.set_3d_properties([Z_m1[frame]])

        body2.set_data([X_m2[frame]], [Y_m2[frame]])
        body2.set_3d_properties([Z_m2[frame]])

        return traj, point, body1, body2

    anim = FuncAnimation(
        fig,
        update,
        frames=len(t),
        interval=interval,
        blit=False
    )

    return anim

# ------------------------------------------------------------
# RUN THE ANIMATION
# (Assumes your arrays are already defined in the notebook)
# ------------------------------------------------------------
# Earth is at the origin in ECI frame
z = np.zeros_like(Y)
X_m1, Y_m1, Z_m1 = z[0], z[1], z[2]
# Spacecraft trajectory is in Y array
X_Iner, Y_Iner, Z_Iner = Y[0], Y[1], Y[2]
# Get Moon orbit
moon_states = np.array([moon_state(ti) for ti in t])
X_m2, Y_m2, Z_m2 = moon_states[:, 0], moon_states[:, 1], moon_states[:, 2]  


anim = animate_3body_trajectory(
    t,
    X_Iner, Y_Iner, Z_Iner,
    X_m1, Y_m1, Z_m1,
    X_m2, Y_m2, Z_m2
)

HTML(anim.to_jshtml())
plt.legend()
plt.show()